# Pandas — IQR Outlier Removal Shortcut

Walking through this elegant two-line outlier removal:

```python
within_bounds = (df[outlier_candidates] >= lower) & (df[outlier_candidates] <= upper)
df = df[within_bounds.all(axis=1)]
```

This filters out **any row where any selected column is an outlier** — across multiple columns at once.

Pure pandas, no loops. Below is the breakdown of *why* it works.

In [2]:
import pandas as pd
import numpy as np

## Setup — Sample Data with Outliers

Multiple numeric columns where some rows are outliers in one or more columns.

In [22]:
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Henry'],
    'age':    [25, 30, 35, 200, 28, 32, -5, 45],     # 200 and -5 are outliers
    'salary': [50000, 60000, 75000, 90000, 55000, 1_000_000, 65000, 70000],  # 1M is outlier
    'score':  [85, 78, 92, 88, 250, 81, 90, 87]      # 250 is outlier
})
df

,name,age,salary,score
0,Alice,25,50000,85
1,Bob,30,60000,78
2,Carol,35,75000,92
3,Dave,200,90000,88
4,Eve,28,55000,250
5,Frank,32,1000000,81
6,Grace,-5,65000,90
7,Henry,45,70000,87


In [ ]:
q1 = df[["age", "salary", "score"]].quantile(.25) 
q3 = df[["age", "salary", "score"]].quantile(.75) 
lower = q1 - 1.5 * (q3-q1)
higher = q3 + 1.5 * (q3-q1)
withinlimit = ( 
    (df[["age", "salary", "score"]] >= lower) &
    (df[["age", "salary", "score"]] <= higher)
)
rows_within_limit = withinlimit.all(axis=1)
df = df[rows_within_limit]
df.reset_index()


,index,name,age,salary,score
0,0,Alice,25,50000,85
1,1,Bob,30,60000,78
2,2,Carol,35,75000,92
3,7,Henry,45,70000,87


In [ ]:
q1 = df.iloc[:,1:4].quantile(.25)
q1
mask = (df.loc[:,"age":"score"] > q1)
mask.all(axis=1)

age       False
salary    False
score     False
dtype: bool

## Step 1 — Compute IQR Bounds

**IQR (Interquartile Range)** = Q3 − Q1.

Standard outlier rule:
```
Lower bound = Q1 − 1.5 × IQR
Upper bound = Q3 + 1.5 × IQR
```

Any value outside `[lower, upper]` is considered an outlier.

In [4]:
# Pick the columns to check for outliers
outlier_candidates = ['age', 'salary', 'score']

Q1 = df[outlier_candidates].quantile(0.25)
Q3 = df[outlier_candidates].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print('Q1:\n', Q1, sep='')
print('\nQ3:\n', Q3, sep='')
print('\nlower bounds:\n', lower, sep='')
print('\nupper bounds:\n', upper, sep='')

Q1:
age          27.25
salary    58750.00
score        84.00
Name: 0.25, dtype: float64

Q3:
age          37.5
salary    78750.0
score        90.5
Name: 0.75, dtype: float64

lower bounds:
age          11.875
salary    28750.000
score        74.250
dtype: float64

upper bounds:
age           52.875
salary    108750.000
score        100.250
dtype: float64


Notice that `lower` and `upper` are **Series indexed by column name** — one bound per column.

```
lower = pd.Series({'age': 17.5, 'salary': 42500, 'score': 72.5})
upper = pd.Series({'age': 53.5, 'salary': 95000, 'score': 100})
```

## Step 2 — The Comparison Step (The Magic)

```python
within_bounds = (df[outlier_candidates] >= lower) & (df[outlier_candidates] <= upper)
```

What happens here?

- `df[outlier_candidates]` is a DataFrame (multiple columns)
- `lower` is a Series indexed by column names
- Comparing **DataFrame ≥ Series** → pandas broadcasts the Series across rows

Each column of the DataFrame is compared to the matching Series element by **column label**.

In [5]:
# Just check the lower bound first
above_lower = df[outlier_candidates] >= lower
above_lower

,age,salary,score
0,True,True,True
1,True,True,True
2,True,True,True
3,True,True,True
4,True,True,True
5,True,True,True
6,False,True,True
7,True,True,True


Each cell is now `True` if it's `>= lower bound for its column`, else `False`.

Notice the broadcasting:
```
For column 'age':    df['age'] >= lower['age'] = df['age'] >= 17.5
For column 'salary': df['salary'] >= lower['salary'] = df['salary'] >= 42500
For column 'score':  df['score'] >= lower['score'] = df['score'] >= 72.5
```

All three comparisons happen in **one expression** thanks to pandas index alignment.

In [ ]:
# Same for upper bound
below_upper = df[outlier_candidates] <= upper
below_upper

In [ ]:
# Combine: must be within BOTH bounds (use & for element-wise AND)
within_bounds = (df[outlier_candidates] >= lower) & (df[outlier_candidates] <= upper)
within_bounds

Now `within_bounds` is a **boolean DataFrame** with the same shape as `df[outlier_candidates]`:

- `True` = value is within bounds (normal)
- `False` = value is an outlier

## Step 3 — Reduce to One Boolean Per Row

We have a DataFrame of booleans — but to filter rows, we need ONE True/False per row.

**Rule:** keep row only if ALL columns are within bounds.

```python
within_bounds.all(axis=1)
```

`axis=1` collapses across columns — for each row, returns `True` only if ALL column values are True.

In [ ]:
row_ok = within_bounds.all(axis=1)
row_ok

`row_ok` is a **Series of booleans** (one per row):

- `True` = all selected columns within bounds → keep row
- `False` = at least one column is an outlier → drop row

### Why `axis=1`?

Remember from `8Axes.md` — the axis you pass is the one that gets COLLAPSED:

```
axis=0 → collapses rows → result PER COLUMN
axis=1 → collapses columns → result PER ROW  ← we want this
```

We want **one True/False per row**, so we collapse the column axis → `axis=1`.

## Step 4 — Apply the Mask

```python
df = df[within_bounds.all(axis=1)]
```

Standard pandas row filtering — keep rows where the mask is True.

In [ ]:
df_clean = df[within_bounds.all(axis=1)]
df_clean

Rows with outliers in any of the checked columns are removed.

In our data:
- Dave (age=200) → removed
- Eve (score=250) → removed
- Frank (salary=1M) → removed
- Grace (age=-5) → removed

## The Full Two-Line Solution

Putting it all together:

In [ ]:
# Reset df
df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Henry'],
    'age':    [25, 30, 35, 200, 28, 32, -5, 45],
    'salary': [50000, 60000, 75000, 90000, 55000, 1_000_000, 65000, 70000],
    'score':  [85, 78, 92, 88, 250, 81, 90, 87]
})

outlier_candidates = ['age', 'salary', 'score']

# Compute bounds
Q1 = df[outlier_candidates].quantile(0.25)
Q3 = df[outlier_candidates].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

# THE SHORTCUT — two lines
within_bounds = (df[outlier_candidates] >= lower) & (df[outlier_candidates] <= upper)
df_clean = df[within_bounds.all(axis=1)]

df_clean

## Why This Is Elegant

Compare to the verbose loop version:

```python
# Verbose version — column by column
mask = pd.Series(True, index=df.index)   # start with all True
for col in outlier_candidates:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask &= (df[col] >= lower) & (df[col] <= upper)
df_clean = df[mask]
```

The shortcut does the same thing in 2 lines instead of 8 — by leveraging:

1. **DataFrame-Series broadcasting** — comparing `df[cols] >= lower` aligns by column name
2. **Element-wise `&`** — combines two boolean DataFrames
3. **`.all(axis=1)`** — reduces row-wise to a single boolean per row
4. **Boolean indexing** — `df[mask]` keeps only matching rows

## Variations

### Use `.any()` instead of `.all()` for OR logic

If you want to keep rows where ANY column is within bounds (rare):

```python
df[within_bounds.any(axis=1)]   # at least one column is normal
```

### Replace outliers instead of dropping rows

Sometimes dropping rows loses too much data. Instead, **clip** them to the bounds:

In [ ]:
# Clip outliers to the bound (winsorisation)
df_clipped = df.copy()
for col in outlier_candidates:
    df_clipped[col] = df_clipped[col].clip(lower[col], upper[col])
df_clipped

### Z-score-based outlier detection (alternative)

Another popular approach — flag values more than 3 standard deviations from the mean:

In [ ]:
from scipy import stats

z_scores = np.abs(stats.zscore(df[outlier_candidates]))
df_z_clean = df[(z_scores < 3).all(axis=1)]
df_z_clean

Same `.all(axis=1)` pattern — pandas idiom for "all columns must satisfy the condition".

### IQR vs Z-score

```
IQR:
   - Robust (uses quartiles, not mean/std)
   - Insensitive to extreme outliers themselves
   - Standard for skewed data
   - Default: 1.5 × IQR

Z-score:
   - Assumes roughly normal distribution
   - Sensitive to existing outliers (they bias mean and std)
   - Simple to compute and explain
   - Default: |z| > 3
```

## Mental Model — The Three Pandas Tricks Combined

This one-liner combines **three** powerful pandas features:

### 1. Broadcasting (Series with DataFrame)

```
DataFrame (n × c)  comparison  Series (c indexed by column)
                    ↓
Series's values broadcast across all ROWS of the DataFrame
→ result is a DataFrame of the same shape, with element-wise comparison
```

### 2. Boolean DataFrame Operations

```
Two boolean DataFrames can be combined with:
   & (and)
   | (or)
   ~ (not)
Element-wise — produces a boolean DataFrame of the same shape.
```

### 3. Axis Reduction with `.all()` / `.any()`

```
axis=1 → collapse columns → one boolean per ROW
axis=0 → collapse rows → one boolean per COLUMN

`.all()` returns True only if ALL elements are True.
`.any()` returns True if AT LEAST ONE element is True.
```

## Common Pitfalls

### Forgetting parentheses with `&`

```python
# WRONG — Python evaluates `>=` and `<=` before `&`, but `&` has lower precedence than these
# Actually for booleans this is OK, but for mixed comparisons watch out
df[outlier_candidates] >= lower & df[outlier_candidates] <= upper   # ambiguous!

# RIGHT — always parenthesise each comparison
(df[outlier_candidates] >= lower) & (df[outlier_candidates] <= upper)
```

### Using `and` instead of `&`

```python
# WRONG — `and` doesn't work element-wise
(df[col1] >= 0) and (df[col2] <= 100)
# Raises: ValueError

# RIGHT — `&` works element-wise on Series/DataFrames
(df[col1] >= 0) & (df[col2] <= 100)
```

### Confusing `axis=0` and `axis=1`

```python
within_bounds.all(axis=0)    # returns one bool PER COLUMN — wrong here!
within_bounds.all(axis=1)    # returns one bool PER ROW — correct
```

Remember: **the axis you pass is the one that disappears**. We want per-row, so disappear columns → `axis=1`.

## Summary

```
Shortcut for multi-column outlier removal:

   1. Compute lower / upper Series (one bound per column)

   2. (df[cols] >= lower) & (df[cols] <= upper)
      → broadcasts Series across rows
      → produces boolean DataFrame: cell-wise within bounds?

   3. .all(axis=1)
      → collapses columns → one bool per row
      → True only if ALL columns are within bounds

   4. df[mask]
      → standard boolean row filter

Combines: broadcasting + element-wise &/| + axis reduction
```

> This pattern appears everywhere in pandas data cleaning. Once you understand the three primitives (broadcast, element-wise boolean ops, `.all()` / `.any()` along an axis), you can write concise pipelines for any multi-column filtering task.